# Openjutsu on Google Colab

Runs ComfyUI, comfy-api-proxy and the Openjutsu app on a Colab GPU, then opens the app through a Cloudflare quick tunnel. Run the cells from top to bottom. The work is done by `scripts/linux/setup.sh` and `scripts/linux/start.sh` in the repository; `docs/linux-gpu.md` explains them.

**Before you start**

- **Untested:** this notebook has not been run on Colab yet.
- **No live progress:** Cloudflare's quick-tunnel documentation says quick tunnels do not support Server-Sent Events, which the app's progress view uses. The page then checks the job every 5 s: the status and the result appear, but not the steps, clip numbers or mask previews. Finished videos are also saved in `/content/ComfyUI/output`.
- **Runtime:** a paid plan (Pro, Pro+ or Pay As You Go) with an **L4** or **A100** GPU: *Runtime → Change runtime type*. Free runtimes get a "standard" memory profile (Google publishes no figure), and H3 was tested with 64 GB of system RAM. Cell 1 shows what this runtime has.
- **Colab's rules:** Colab's [FAQ](https://research.google.com/colaboratory/faq.html) lists *creating deepfakes* and "web service offerings not related to interactive compute" among the activities disallowed on all managed runtimes, paid ones included. Swapping a person in a video can count as the first. Decide for yourself, and only use videos and photos of people who agreed.
- **Downloads every session:** the runtime's disk is wiped when it ends, so each new runtime downloads the models again: about 51 GB, plus about 30 GB with From picture.
- **Upload size:** the app accepts up to 512 MB per job. Whether a quick tunnel has a lower limit is not documented on Cloudflare's quick-tunnel page; if a large upload fails, try a shorter video.
- **Secrets** (the key icon on the left; allow this notebook to read them):
  - `OPENJUTSU_PASSWORD`: 12 or more characters. Without it, a password is generated and shown below, and it stays in the notebook's output.
  - `HF_TOKEN`: only for From picture. Accept the licence at [black-forest-labs/flux-3-image-klein-comfy](https://huggingface.co/black-forest-labs/flux-3-image-klein-comfy) first, then create a read token.

In [ ]:
#@title Settings
REPO = "https://github.com/antorio/openjutsu"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
#@markdown Fix faces adds about 0.5 GB. From picture adds about 30 GB (23 GB with rank 64) and needs the `HF_TOKEN` secret.
WITH_FACE = True  #@param {type:"boolean"}
WITH_PICTURE = False  #@param {type:"boolean"}
SOLAR_RANK = "384"  #@param ["384", "64"]

In [ ]:
#@title 1. Check the runtime
import os, shutil, subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode:
    raise RuntimeError("No GPU. Choose Runtime > Change runtime type > L4 or A100.")
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
disk = shutil.disk_usage("/content").free / 2**30
need = 51 + (0.5 if WITH_FACE else 0) + ((23 if SOLAR_RANK == "64" else 30) if WITH_PICTURE else 0) + 15
print(f"GPU:       {gpu.stdout.strip()}")
print(f"RAM:       {ram:.0f} GB")
print(f"Free disk: {disk:.0f} GB (these settings need about {need:.0f} GB)")
if ram < 48:
    print("WARNING: less than 48 GB of RAM. H3 was tested with 64 GB; expect out-of-memory errors.")
if disk < need:
    print("WARNING: not enough disk. Turn off From picture or choose rank 64.")

In [ ]:
#@title 2. Install (run again after the runtime restarts)
import os, subprocess

if os.path.isdir("/content/openjutsu/.git"):
    subprocess.run(["git", "-C", "/content/openjutsu", "pull", "-q", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO, "/content/openjutsu"], check=True)
os.environ.update(OPENJUTSU_HOME="/content", WITH_FACE=str(int(WITH_FACE)), WITH_PICTURE=str(int(WITH_PICTURE)), SOLAR_RANK=SOLAR_RANK)
if WITH_PICTURE:
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as error:
        raise RuntimeError("From picture needs the HF_TOKEN secret (key icon on the left).") from error
!bash /content/openjutsu/scripts/linux/setup.sh

In [ ]:
#@title 3. Start Openjutsu
import os, secrets
from google.colab import userdata

try:
    password = userdata.get("OPENJUTSU_PASSWORD")
except Exception:
    password = None
if password and len(password) < 12:
    raise RuntimeError("The OPENJUTSU_PASSWORD secret must have 12 or more characters.")
if not password:
    password = secrets.token_urlsafe(18)
    print(f"No OPENJUTSU_PASSWORD secret, so this session's password is: {password}")
os.environ.update(OPENJUTSU_HOME="/content", OPENJUTSU_PASSWORD=password, TUNNEL="cloudflared")
!bash /content/openjutsu/scripts/linux/start.sh

In [ ]:
#@title 4. Keep the runtime busy
#@markdown Colab ends idle runtimes without saying what counts as idle; a running cell is meant to keep this one busy, but that is not guaranteed. It runs until you stop it (■). It shows the address and the latest ComfyUI line, and stops if a service has stopped.
import os, time
from IPython.display import clear_output

logs = "/content/logs"
address = open(f"{logs}/tunnel.url").read().strip()

def alive(name):
    try:
        os.kill(int(open(f"{logs}/{name}.pid").read()), 0)
        return True
    except (OSError, ValueError):
        return False

def last_line(name):
    with open(f"{logs}/{name}.log", errors="replace") as log:
        lines = log.read()[-4000:].replace("\r", "\n").strip().splitlines()
    return lines[-1] if lines else ""

try:
    while True:
        stopped = [name for name in ("comfyui", "proxy", "app", "tunnel") if not alive(name)]
        clear_output(wait=True)
        print(f"Openjutsu: {address}  (user: openjutsu)")
        print(time.strftime("%H:%M:%S"), "ComfyUI:", last_line("comfyui")[:200])
        if stopped:
            for name in stopped:
                print(f"\n{name} stopped. Last lines of {logs}/{name}.log:")
                print(open(f"{logs}/{name}.log", errors="replace").read()[-3000:])
            break
        time.sleep(30)
except KeyboardInterrupt:
    print("Stopped watching. Openjutsu keeps running until the runtime disconnects.")

In [ ]:
#@title 5. Stop Openjutsu
!bash /content/openjutsu/scripts/linux/start.sh stop